In [ ]:
import asyncio
import pandas as pd

from rds_db_setup import get_or_create_async_db_engine # db_setup.py 경로에 맞게 수정

from typing import List

import langchain
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_postgres.vectorstores import PGVector
from langchain.chains.query_constructor.base import AttributeInfo
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain.retrievers.self_query.base import SelfQueryRetriever
DB_NAME = "hy_rag_db"
QNA_COLLECTION_NAME = "qna_documents"
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(temperature=0, model="gpt-5-mini")

db_engine = await get_or_create_async_db_engine(DB_NAME)
vectorstore = PGVector(
    embeddings=embedding_model,
    collection_name=QNA_COLLECTION_NAME,
    connection=db_engine,
    use_jsonb=True, # 메타데이터 검색 성능을 위해 권장
)

# 테스트용 쿼리들
test_queries = [
    # 1. 단순 정보 확인형 쿼리
    "12년 특례는 원서 몇 번 쓸 수 있어?",
    "수시 지원 횟수 제한 알려줘.",
    "3년 특례도 6번만 지원 가능해?",

    # 2. 메타데이터 필터링 테스트용 쿼리
    "3년 특례랑 12년 특례 전형의 지원 횟수 제한에 대해 알려줘.",
    "전형 이해 좀 하고 싶은데, 지원 횟수 제한이 제일 궁금해.",
    "수시 전형 지원 횟수 관련해서 설명해줘.",

    # 3. 비교 및 추론형 쿼리
    "3년 특례랑 12년 특례의 가장 큰 차이점이 뭐야?",
    "나는 원서 개수 제한 없이 마음껏 지원하고 싶은데, 어떤 전형을 노려야 할까?",
    "수시랑 재외국민 전형은 지원 횟수를 합산해서 계산하는 거야?",

    # 4. 잘못된 정보 확인 및 예외 케이스 쿼리
    "2025학년도부터 수시 6회 제한 폐지된 거 맞지?",
    "입시 제도 개편돼서 이제 수시 아무데나 써도 된다던데?",
    "12년 특례 정원 외 0.5% 삭제가 무슨 뜻이야?",
]

'hy_rag_db' 데이터베이스에 'vector' 확장이 활성화되었습니다.


In [ ]:
'''
SelfQueryRetriever 메타데이터 검색 활용
- 시간이 굉장히 오래걸림...
'''

# 메타데이터 필드 정보 정의
metadata_field_info = [
    AttributeInfo(
        name="category1",
        description="입시 전형의 대분류. '수시', '정시', '편입' 중 하나일 수 있음.",
        type="string",
    ),
    AttributeInfo(
        name="category2",
        description="입시 전형의 중분류. 예를 들어 '학생부교과', '학생부종합', '논술' 등이 있음.",
        type="string",
    ),
    AttributeInfo(
        name="category3",
        description="입시 전형의 소분류. 상세 전형명이 포함됨.",
        type="string",
    ),
    AttributeInfo(
        name="source",
        description="문서의 출처. 현재는 '질문정리 FAQ'만 있음.",
        type="string",
    ),
]

document_content_description = "학생들의 입시 관련 질문과 그에 대한 답변"

# 위 값으로 메타데이터 필터하는 검색기
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    search_kwargs={"filter": {"source": "질문정리 FAQ"}, "k": 8},
    verbose=True    # 아래 코드 활성화하면 ainvoke실행시킬 때 디버깅 나옴
)
# langchain.debug = True
# langchain.debug = False

# 위 기준으로 문서 선택
filtered_docs = await retriever.ainvoke(test_queries[0])
if filtered_docs:
    print(f"\n--- 📚 검색된 문서 ({len(filtered_docs)}개) ---")
    for i, doc in enumerate(filtered_docs):
        print(f"\n--- 결과 {i+1} ---")
        print(f"내용: {doc.page_content[:100]}...")
        print(f"메타데이터: {doc.metadata}")
else:
    print("검색된 문서가 없습니다.")


# retriever가 반환한 Document 리스트를 하나의 문자열로 합쳐주는 함수
def format_docs(docs):
    return "\n\n".join(
        d.page_content if hasattr(d, "page_content") else str(d)
        for d in (docs or [])
    )

# LLM에게 전달할 프롬프트 템플릿 정의
template = """
당신은 대학교 입시 질문에 답변하는 친절한 AI 어시스턴트입니다.
제공된 컨텍스트 정보를 바탕으로 사용자의 질문에 답변하세요. 
컨텍스트에 없는 내용은 답변하지 마세요.

컨텍스트:
{context}

질문:
{question}

답변:
"""
prompt = ChatPromptTemplate.from_template(template)

# 프롬프트 체인 구성

rag_chain = {
    "context": retriever | format_docs,      # retriever 출력 -> 포맷팅
    "question": RunnablePassthrough()
} | prompt | llm  | StrOutputParser()

print(f"사용자 질문: {test_queries[0]}")

# 체인을 실행하면 retriever로 문서를 찾고, 그 결과를 LLM에 넘겨 최종 답변을 생성합니다.
final_answer = await rag_chain.ainvoke(test_queries[0])

print("\n--- 🤖 AI 답변 ---")
print(final_answer)


간단히: 제한이 없습니다. 12년 특례(전 교육과정 해외이수)는 지원 횟수 제한이 삭제되어 여러 대학에 제한 없이 지원할 수 있습니다.

주의사항(중요)
- 대학별 전형 일정이 달라 겹칠 수 있으니 일정 충돌이 없도록 전략적으로 지원해야 합니다.
- 동일 대학 내에서는 원칙적으로 1개 모집단위(전공)만 지원 가능(예외: 연세대 12년특례는 일반학과 1개 + 글로벌인재학부 1개 동시 지원 가능).
- 수시·3년특례는 여전히 6회 제한 적용되므로 혼동하지 마세요.
- 반드시 지원할 학년도 각 대학의 모집요강으로 최종 확인하세요.
